# GDS Lab: Node Similarity — Aircraft Fault Profiles

> **Optional GDS Example.** This notebook is an optional example demonstrating the Neo4j Graph Data Science (GDS) library. It is not required to complete the workshop. Run it if you want to explore graph algorithms on the aircraft dataset.

This notebook uses the Node Similarity algorithm to find aircraft that share
overlapping fault-type portfolios. The output is a set of `SIMILAR_FAULT_PROFILE`
relationships, queryable in both Neo4j Browser and Databricks.

## What You'll Learn
- How to enrich a graph with derived nodes to enable meaningful similarity
- How Jaccard similarity works on bipartite graphs
- The difference between Node Similarity (neighborhood-based) and kNN (property-based)
- How to read a similarity sub-graph back into Databricks

## The Graph Enrichment Pattern
Aircraft in the original graph connect to `Component` nodes — but every component
is unique to one aircraft, so raw neighborhood similarity would be zero across
aircraft boundaries. We fix this by creating **`FaultType` nodes** that represent
distinct fault-severity combinations (e.g., `bearing wear_MAJOR`). Multiple aircraft
can connect to the same FaultType, making cross-aircraft similarity computable.

**Jaccard similarity** = `|shared fault types| / |all fault types for the pair|`

Two aircraft with identical fault portfolios score 1.0. Two aircraft with no shared
fault types score 0.0.

## Prerequisites
- Neo4j Aura credentials (AuraDB Professional or higher)
- Full dataset loaded via `01_aircraft_etl_to_neo4j.ipynb`

## Instructions
1. Enter credentials in Section 1
2. Run all cells in order — Section 7 cleans up the temporary FaultType nodes

## Section 1: Configuration

In [ ]:
%pip install neo4j

In [ ]:
# ============================================
# CONFIGURATION
# ============================================

NEO4J_URI      = ""      # e.g., "neo4j+s://xxxxxxxx.databases.neo4j.io"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = ""
NEO4J_DATABASE = "neo4j"  # Neo4j database to use (Aura default is "neo4j")

if not NEO4J_URI or not NEO4J_PASSWORD:
    print("WARNING: Please enter your Neo4j credentials above before running!")
else:
    print(f"Configuration ready — {NEO4J_URI}")

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

def run_query(cypher, params=None):
    """Execute a Cypher query and return results as a list of dicts."""
    with driver.session(database=NEO4J_DATABASE) as session:
        return [dict(r) for r in session.run(cypher, params or {})]

spark.conf.set("neo4j.url", NEO4J_URI)
spark.conf.set("neo4j.authentication.basic.username", NEO4J_USERNAME)
spark.conf.set("neo4j.authentication.basic.password", NEO4J_PASSWORD)
spark.conf.set("neo4j.database", NEO4J_DATABASE)

def spark_query(cypher):
    """Read Neo4j query results as a Spark DataFrame."""
    return (spark.read
        .format("org.neo4j.spark.DataSource")
        .option("query", cypher)
        .load())

result = run_query("RETURN 'Connected to Neo4j' AS status")
print(result[0]["status"])

## Section 2: Explore Fault Type Diversity

Node Similarity works best when nodes share a meaningful number of neighbors.
First confirm the fault vocabulary is rich enough for interesting overlaps.

In [ ]:
# How many distinct fault-type + severity combinations exist?
vocabulary = run_query("""
    MATCH (m:MaintenanceEvent)
    WITH m.fault + '_' + m.severity AS fault_key, m.fault AS fault, m.severity AS severity
    RETURN fault_key, fault, severity, count(*) AS occurrences
    ORDER BY occurrences DESC
""")

print(f"Distinct fault-type × severity combinations: {len(vocabulary)}\n")
print(f"{'Fault Key':<36} {'Occurrences':>12}")
print("-" * 50)
for r in vocabulary:
    print(f"{r['fault_key']:<36} {r['occurrences']:>12}")

In [ ]:
# How many distinct fault types has each aircraft experienced?
# Diversity per aircraft determines how much similarity information we have per node.
per_aircraft = run_query("""
    MATCH (a:Aircraft)-[:HAS_SYSTEM]->(:System)-[:HAS_COMPONENT]->(:Component)
          -[:HAS_EVENT]->(m:MaintenanceEvent)
    RETURN a.tail_number   AS tail_number,
           a.model         AS model,
           count(DISTINCT m.fault + '_' + m.severity) AS distinct_fault_types,
           count(m)                                   AS total_events
    ORDER BY distinct_fault_types DESC
""")

print(f"{'Aircraft':<12} {'Model':<12} {'Fault Types':>12} {'Total Events':>14}")
print("-" * 52)
for r in per_aircraft:
    print(f"{r['tail_number']:<12} {r['model']:<12} {r['distinct_fault_types']:>12} {r['total_events']:>14}")

## Section 3: Create FaultType Nodes (Graph Enrichment)

We create temporary `FaultType` nodes and `EXPERIENCED_FAULT` relationships.
This makes the bipartite graph explicit in Neo4j, which is required for the
native projection in Section 4.

Each FaultType node represents one `fault + severity` combination. All aircraft
that experienced `bearing wear_MAJOR` will connect to the same FaultType node,
making Jaccard similarity across aircraft computable.

> **Note:** These FaultType nodes are temporary scaffolding. Section 7 removes them.

In [ ]:
# Create FaultType nodes and connect Aircraft via EXPERIENCED_FAULT
result = run_query("""
    MATCH (a:Aircraft)-[:HAS_SYSTEM]->(:System)-[:HAS_COMPONENT]->(:Component)
          -[:HAS_EVENT]->(m:MaintenanceEvent)
    WITH a, m.fault + '_' + m.severity AS fault_key,
            m.fault                     AS fault_name,
            m.severity                  AS severity
    MERGE (ft:FaultType {key: fault_key})
        ON CREATE SET ft.fault = fault_name, ft.severity = severity
    MERGE (a)-[:EXPERIENCED_FAULT]->(ft)
    RETURN count(DISTINCT ft) AS fault_type_nodes_created,
           count(DISTINCT a)  AS aircraft_connected
""")

r = result[0]
print(f"FaultType nodes created: {r['fault_type_nodes_created']}")
print(f"Aircraft connected:      {r['aircraft_connected']}")

In [ ]:
# Verify the bipartite structure — sample aircraft and their fault type connections
sample = run_query("""
    MATCH (a:Aircraft)-[:EXPERIENCED_FAULT]->(ft:FaultType)
    WHERE a.tail_number IN ['N10000', 'N10001']
    RETURN a.tail_number AS aircraft,
           collect(ft.key) AS fault_types
    ORDER BY aircraft
""")

for r in sample:
    print(f"{r['aircraft']}:")
    for ft in sorted(r["fault_types"]):
        print(f"  → {ft}")

## Section 4: Project the Bipartite Graph and Run Node Similarity

In [ ]:
run_query("CALL gds.graph.drop('aircraft-faulttype', false) YIELD graphName")
print("Cleared previous projection.")

In [ ]:
# Native projection of the bipartite Aircraft-FaultType graph
result = run_query("""
    CALL gds.graph.project(
        'aircraft-faulttype',
        ['Aircraft', 'FaultType'],
        {EXPERIENCED_FAULT: {orientation: 'NATURAL'}}
    )
    YIELD graphName, nodeCount, relationshipCount
""")

proj = result[0]
print(f"Projection '{proj['graphName']}':")
print(f"  Nodes:         {proj['nodeCount']}  (Aircraft + FaultType)")
print(f"  Relationships: {proj['relationshipCount']}")

In [ ]:
# Run Node Similarity and filter to Aircraft-Aircraft pairs only
# topK: 5 returns the 5 most similar peers per aircraft
# similarityCutoff: 0.2 excludes pairs with very little overlap
similarity_results = run_query("""
    CALL gds.nodeSimilarity.stream('aircraft-faulttype', {
        topK: 5,
        similarityCutoff: 0.2
    })
    YIELD node1, node2, similarity
    WHERE gds.util.asNode(node1):Aircraft
      AND gds.util.asNode(node2):Aircraft
    RETURN gds.util.asNode(node1).tail_number AS aircraft_a,
           gds.util.asNode(node1).model       AS model_a,
           gds.util.asNode(node2).tail_number AS aircraft_b,
           gds.util.asNode(node2).model       AS model_b,
           round(similarity, 4)               AS jaccard_similarity
    ORDER BY jaccard_similarity DESC
    LIMIT 20
""")

print(f"Top aircraft pairs by Jaccard similarity:\n")
print(f"{'Aircraft A':<12} {'Model A':<12} {'Aircraft B':<12} {'Model B':<12} {'Jaccard':>10}")
print("-" * 62)
for r in similarity_results:
    cross_model = " ◀" if r["model_a"] != r["model_b"] else ""
    print(f"{r['aircraft_a']:<12} {r['model_a']:<12} {r['aircraft_b']:<12} {r['model_b']:<12} {r['jaccard_similarity']:>10}{cross_model}")

cross = sum(1 for r in similarity_results if r["model_a"] != r["model_b"])
print(f"\n◀ = cross-model pair ({cross} of {len(similarity_results)} shown)")

## Section 5: Write SIMILAR_FAULT_PROFILE Relationships

Persisting the similarity scores as relationships makes them queryable via Cypher
and visible in the Neo4j Browser graph visualization.

In [ ]:
result = run_query("""
    CALL gds.nodeSimilarity.write('aircraft-faulttype', {
        topK: 5,
        similarityCutoff: 0.2,
        writeRelationshipType: 'SIMILAR_FAULT_PROFILE',
        writeProperty: 'jaccard_score'
    })
    YIELD nodesCompared, relationshipsWritten
""")

r = result[0]
print(f"Compared {r['nodesCompared']} node pairs")
print(f"Wrote {r['relationshipsWritten']} SIMILAR_FAULT_PROFILE relationships")

In [ ]:
# Verify and sample the written relationships
verify = run_query("""
    MATCH (a:Aircraft)-[r:SIMILAR_FAULT_PROFILE]->(b:Aircraft)
    RETURN a.tail_number   AS aircraft,
           a.model         AS model,
           b.tail_number   AS peer,
           b.model         AS peer_model,
           round(r.jaccard_score, 4) AS jaccard_score
    ORDER BY jaccard_score DESC
    LIMIT 10
""")

print(f"{'Aircraft':<12} {'Model':<12} {'Peer':<12} {'Peer Model':<12} {'Jaccard':>10}")
print("-" * 62)
for r in verify:
    cross = " ◀" if r["model"] != r["peer_model"] else ""
    print(f"{r['aircraft']:<12} {r['model']:<12} {r['peer']:<12} {r['peer_model']:<12} {r['jaccard_score']:>10}{cross}")

## Section 6: Inspect the Similarity Network in Databricks

Pull the similarity network into Databricks and compute community-level summaries.

In [ ]:
# Read the full similarity network
similarity_df = spark_query("""
    MATCH (a:Aircraft)-[r:SIMILAR_FAULT_PROFILE]->(b:Aircraft)
    RETURN a.tail_number   AS aircraft,
           a.model         AS model,
           a.operator      AS operator,
           b.tail_number   AS peer,
           b.model         AS peer_model,
           round(r.jaccard_score, 4) AS jaccard_score
    ORDER BY aircraft, jaccard_score DESC
""")

display(similarity_df)

In [ ]:
from pyspark.sql.functions import col

# Cross-model similar pairs — the most operationally interesting result.
# These aircraft share failure modes despite being different models.
cross_model = (similarity_df
    .filter(col("model") != col("peer_model"))
    .orderBy("jaccard_score", ascending=False))

count = cross_model.count()
print(f"Cross-model similar pairs: {count}")
display(cross_model)

In [ ]:
# For the most similar cross-model pair, show the shared fault types
top_pair = cross_model.first()
if top_pair:
    shared_faults = run_query("""
        MATCH (a:Aircraft {tail_number: $ac1})-[:EXPERIENCED_FAULT]->(ft:FaultType)
              <-[:EXPERIENCED_FAULT]-(b:Aircraft {tail_number: $ac2})
        RETURN ft.fault    AS fault_type,
               ft.severity AS severity,
               ft.key      AS fault_key
        ORDER BY fault_key
    """, {"ac1": top_pair["aircraft"], "ac2": top_pair["peer"]})

    print(f"{top_pair['aircraft']} ({top_pair['model']}) "
          f"↔ {top_pair['peer']} ({top_pair['peer_model']}) "
          f"— Jaccard: {top_pair['jaccard_score']}")
    print(f"\nShared fault types ({len(shared_faults)}):")
    for r in shared_faults:
        print(f"  {r['fault_type']:<28} {r['severity']}")
else:
    print("No cross-model pairs found — all similar pairs are within the same model.")

## Section 7: Clean Up

Remove the temporary FaultType nodes and EXPERIENCED_FAULT relationships.
The `SIMILAR_FAULT_PROFILE` relationships between Aircraft nodes are retained.

In [ ]:
# Drop the in-memory projection
run_query("CALL gds.graph.drop('aircraft-faulttype', false) YIELD graphName")

# Remove temporary FaultType nodes and EXPERIENCED_FAULT relationships
result = run_query("""
    MATCH (ft:FaultType)
    DETACH DELETE ft
    RETURN count(*) AS deleted
""")

driver.close()
print(f"Cleaned up temporary FaultType nodes.")
print("SIMILAR_FAULT_PROFILE relationships remain on Aircraft nodes.")

## Neo4j Browser Queries

**Visualize the aircraft similarity network:**
```cypher
MATCH (a:Aircraft)-[r:SIMILAR_FAULT_PROFILE]->(b:Aircraft)
RETURN a, r, b
```

**Most similar aircraft pairs:**
```cypher
MATCH (a:Aircraft)-[r:SIMILAR_FAULT_PROFILE]->(b:Aircraft)
RETURN a.tail_number AS aircraft, a.model AS model,
       b.tail_number AS peer, b.model AS peer_model,
       round(r.jaccard_score, 4) AS jaccard_score
ORDER BY jaccard_score DESC
LIMIT 15
```

**Compare with kNN peers (requires notebook 03 to have run):**
```cypher
MATCH (a:Aircraft {tail_number: 'N10000'})
OPTIONAL MATCH (a)-[r1:SIMILAR_FAULT_PROFILE]->(peer1:Aircraft)
OPTIONAL MATCH (a)-[r2:SIMILAR_PROFILE]->(peer2:Aircraft)
RETURN peer1.tail_number AS jaccard_peer, round(r1.jaccard_score, 4) AS jaccard_score,
       peer2.tail_number AS knn_peer,     round(r2.similarity_score, 4) AS knn_score
ORDER BY jaccard_score DESC
```

**Aircraft that appear as peers in both similarity algorithms:**
```cypher
MATCH (a:Aircraft)-[:SIMILAR_FAULT_PROFILE]->(peer:Aircraft)
MATCH (a)-[:SIMILAR_PROFILE]->(peer)
RETURN a.tail_number AS aircraft, peer.tail_number AS peer,
       a.model AS model, peer.model AS peer_model
ORDER BY aircraft
```